<a href="https://colab.research.google.com/github/EnriqueDiaz2/LLM-Rag/blob/main/LLM_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
!pip install FlagEmbedding==1.3.3

  Using cached flagembedding-1.3.3-py3-none-any.whl
  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
  Using cached datasets-2.19.0-py3-none-any.whl.metadata (19 kB)
  Using cached pyarrow_hotfix-0.7-py3-none-any.whl.metadata (3.6 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached fsspec-2024.3.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.19.1.tar.gz (321 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
  Using cached multiprocess-0.70.18-py313-none-any.whl.metadata (7.2 kB)
  Using cached multiprocess-0.70.17-py313-none-any.whl.metadata (7.2 kB)
  Using cached multiproce

In [76]:
!pip install --upgrade pymilvus "pymilvus[model]" langchain-community  pdfplumber

In [77]:
!pip install milvus_model

In [78]:
# Cargar y leer el contenido de un PDF
from langchain_community.document_loaders import PDFPlumberLoader
loader = PDFPlumberLoader("/content/Inteligencia_Artificial.pdf")
docs = loader.load()
print("Número de páginas en el PDF:", len(docs))

# Extraer el texto de cada página para embeberlo (vectorizar)
doc_texts = [page.page_content for page in docs]
print("Texto del documento",doc_texts)

Número de páginas en el PDF: 30
Texto del documento ['Artificial intelligence and privacy\nReport, January 2018\n', '2\n', '3\nContents\nABOUT THIS REPORT ................................................................................................................................................. 4\nLegal sources and use of terminology ............................................................................................................... 4\nARTIFICIAL INTELLIGENCE AND DATA PROTECTION .....................................................................................................5\nHOW DOES ARTIFICIAL INTELLIGENCE WORK? ............................................................................................................ 7\nMachine learning ................................................................................................................................................. 7\nResults of learning ...........................................................

In [79]:
from milvus_model.hybrid import BGEM3EmbeddingFunction
### Crear una instancia para el embebido
# se puede trabajar con CPU pero no es recomendable. es 10x más lento que con GPU
#ef = BGEM3EmbeddingFunction(use_fp16=False, device="cpu")
# Es recomendable usar el GPU
ef = BGEM3EmbeddingFunction(use_fp16=False, device="cuda:0")
dense_dim = ef.dim["dense"]

# Genera embeddings usando el modelo BGE-M3
docs_embeddings = ef(doc_texts)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:03<00:00,  1.89s/it]


In [80]:
"""
Compressed Sparse Row (CSR) es un formato de almacenamiento que se utiliza para
matrices donde la mayoría de los elementos son cero.
En lugar de almacenar todos los ceros (lo que sería un desperdicio de memoria),
el formato CSR solo almacena los valores distintos de cero y sus ubicaciones (índices),
ahorrando una gran cantidad de espacio y mejorando la velocidad de ciertas operaciones
(como el producto matriz-vector o la segmentación por filas).

---with 3150 stored elements and shape (17, 250002)
Lo anterior significa que solo almacena 3,150 elementos en lugar de 17 páginas con
un vocabulario de 250,002
"""

print(docs_embeddings)

{'dense': [array([-0.01992327,  0.04065594, -0.03206449, ...,  0.03012572,
       -0.04144548, -0.00074733], shape=(1024,), dtype=float32), array([-0.006603  ,  0.05431961, -0.0424584 , ..., -0.00874209,
        0.02160891,  0.01301234], shape=(1024,), dtype=float32), array([-0.0269093 ,  0.01215669, -0.01821233, ...,  0.01273115,
       -0.0096347 ,  0.00504806], shape=(1024,), dtype=float32), array([-0.0053495 ,  0.01827344, -0.0415272 , ..., -0.00552993,
       -0.00417259, -0.022278  ], shape=(1024,), dtype=float32), array([ 0.00551084,  0.02008935, -0.00919568, ..., -0.01111286,
        0.01382383, -0.02978063], shape=(1024,), dtype=float32), array([-0.01068686, -0.03723912, -0.04428473, ..., -0.02444351,
        0.03058884, -0.04973668], shape=(1024,), dtype=float32), array([-0.01537715, -0.00759101, -0.02469388, ...,  0.01283922,
        0.03106194, -0.02893026], shape=(1024,), dtype=float32), array([-0.04759919, -0.02506023, -0.05304147, ..., -0.01334016,
        0.01788048,  0

In [81]:
# Cantidad de dimensiones para representar cada palabra en vectorización densa
dense_dim

1024

In [82]:
# Importaremos las clases necesarias para gestionar la conexión con el servidor de Milvus (local o remoto).
# connections - Se utiliza para establecer, desconectar y administrar las sesiones.
# utility - Proporciona funciones de utilidad para la administración de alto nivel, como verificar si una colección existe
# FieldSchema - Se utiliza para especificar el nombre, tipo de datos y si es la clave primaria.
# CollectionSchema - Una clase para definir el esquema completo de una colección. Agrupa múltiples objetos FieldSchema
# DataType - Una enumeración que define los tipos de datos permitidos para los campos de una colección
#            (e.g., DataType.INT64, DataType.FLOAT_VECTOR).  Es crucial para especificar el tipo de los embeddings.
# Collection - La clase principal que representa una colección (el contenedor de documentos/vectores).
#              Se utiliza para crear colecciones, insertar datos, hacer búsquedas vectoriales y borrar colecciones.

from pymilvus import (
    connections,
    utility,
    FieldSchema,
    CollectionSchema,
    DataType,
    Collection,
)

In [83]:
# La conexión será local por lo que instalamos milvus_lite el cual crea un archivo "./milvus.db"
!pip install "pymilvus[milvus_lite]"

In [84]:
# Conectar a Milvus dada la URI
connections.connect(uri="./milvus.db")

# Especificar el esquema de datos para la nueva Colección
fields = [
    # Usa "auto generated id" como clave primaria
    # Se usa "pk" como abreviatura de Primary Key (Clave Primaria).
    # auto_id - Indica que la base de datos generará automáticamente el valor de la clave principal ("pk") cuando se inserte un nuevo registro
    # max_length - longitud máxima de la cadena de texto que se puede almacenar en este campo VARCHAR
    FieldSchema(
        name="pk", dtype=DataType.VARCHAR, is_primary=True, auto_id=True, max_length=100
    ),
    # Almacenar el texto original para recuperarlo en función de la distancia semántica
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096),
    # Milvus ahora admite vectores dispersos y densos.
    # Podemos almacenar cada uno en un campo separado para realizar una búsqueda híbrida en ambos vectores.
    FieldSchema(name="sparse_vector", dtype=DataType.SPARSE_FLOAT_VECTOR),
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=dense_dim),
]
schema = CollectionSchema(fields)

/tmp/ipykernel_811/2882363815.py:2: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(uri="./milvus.db")


In [85]:
# Crear colección (eliminar la antigua si existe)
col_name = "hybrid_demo"
if utility.has_collection(col_name):
    Collection(col_name).drop()

# Reconstruimos los campos para corregir el esquema de forma segura sin modificar la celda anterior
corrected_fields = [
    # Corregimos pk para que sea DataType.INT64 dado que usa auto_id=True
    FieldSchema(name="pk", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096),
    FieldSchema(name="sparse_vector", dtype=DataType.SPARSE_FLOAT_VECTOR),
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=dense_dim)
]
schema = CollectionSchema(corrected_fields)

col = Collection(col_name, schema, consistency_level="Strong")

# Para que la búsqueda vectorial sea eficiente, necesitamos crear índices para los campos vectoriales
sparse_index = {"index_type": "SPARSE_INVERTED_INDEX", "metric_type": "IP"}
col.create_index("sparse_vector", sparse_index)
dense_index = {"index_type": "AUTOINDEX", "metric_type": "IP"}
col.create_index("dense_vector", dense_index)
col.load()

/tmp/ipykernel_811/2714709610.py:3: PyMilvusDeprecationWarning: `utility.has_collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  if utility.has_collection(col_name):
/tmp/ipykernel_811/2714709610.py:4: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  Collection(col_name).drop()
/tmp/ipykernel_811/2714709610.py:4: PyMilvusDeprecationWarning: `Collection.drop` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  Collection(col_name).drop()
/tmp/ipykernel_811/2714709610.py:16: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  col = Collection(col_name, schema, consistency_level="Strong")
/tmp/ipykernel_811/2714709610.py:20: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be re

In [86]:
# Para mayor eficiencia, procesamos en lotes/bloques de 50 elementos.
for i in range(0, len(doc_texts), 50):
    # Extraer el texto en crudo a partir de los objetos del documento
    batched_texts = [doc.page_content for doc in docs[i : i + 50]]  # Extraer el contenido del texto

    # Asegurar que ningún texto exceda el límite físico de VARCHAR de 4096 caracteres configurado en el esquema
    batched_texts = [text[:4096] for text in batched_texts]

    batched_sparse_vectors = docs_embeddings["sparse"][i : i + 50]
    batched_dense_vectors = docs_embeddings["dense"][i : i + 50]

    # Preparar las entidades agrupadas para su inserción
    batched_entities = [
        batched_texts,          # Texto
        batched_sparse_vectors, # Sparse vector embeddings
        batched_dense_vectors,  # Dense vector embeddings
    ]

    # Insertar las entidades dentro de Milvus
    col.insert(batched_entities)

print("Número de entidades insertadas:", col.num_entities)

Número de entidades insertadas: 30


/tmp/ipykernel_811/1911022930.py:20: PyMilvusDeprecationWarning: `Collection.insert` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  col.insert(batched_entities)
/tmp/ipykernel_811/1911022930.py:22: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print("Número de entidades insertadas:", col.num_entities)


In [87]:
# Inserta la consulta
query = input("Inserta tu consulta: ")
print(query)

# Se genera el embebido de la consulta
query_embeddings = ef([query])
# print(query_embeddings)

Inserta tu consulta: 5
5


In [88]:
# AnnSearchRequest - Permite una búsqueda en Milvus con el algoritmo de Vecinos Más Cercanos
# (Approximate Nearest Neighbor (ANN))
# WeightedRanker -
# Un mecanismo de reordenamiento utilizado para combinar los resultados de múltiples solicitudes de búsqueda o etapas de recuperación.
# En un escenario de búsqueda complejo (como una búsqueda híbrida que combina búsqueda vectorial y filtrado escalar),

from pymilvus import (
    AnnSearchRequest,
    WeightedRanker,
)

In [89]:
# - [query_dense_embedding]: Se pasa la consulta como una lista de vectores.
#   Esto permite buscar con uno o varios vectores a la vez.
# - anns_field="dense_vector": Especifica el campo de la colección donde están
#   almacenados los vectores a buscar. En este caso, se asume que el campo se llama "dense_vector".
# - limit=: Pasa el valor del parámetro limit de la función (por defecto 5)
#   para indicar cuántos resultados se quieren obtener.
# - output_fields=["text"]: Especifica que, además de la ID y la distancia,
#   se debe recuperar el valor del campo "text" asociado a cada vector encontrado.
# - param=search_params: Aplica los parámetros de búsqueda definidos anteriormente (la métrica IP).
# - [0]: El método col.search devuelve una lista de resultados, uno por cada vector
#   de consulta y nos quedamos con el primero [0].


def dense_search(col, query_dense_embedding, limit=5):
    search_params = {"metric_type": "IP", "params": {}}
    res = col.search(
        [query_dense_embedding],
        anns_field="dense_vector",
        limit=limit,
        output_fields=["text"],
        param=search_params,
    )[0]
    return [hit.get("text") for hit in res]

In [90]:
def sparse_search(col, query_sparse_embedding, limit=5):
    search_params = {
        "metric_type": "IP", "params": {}, }
    res = col.search(
        [query_sparse_embedding],
        anns_field="sparse_vector",
        limit=limit,
        output_fields=["text"],
        param=search_params,
    )[0]
    return [hit.get("text") for hit in res]

In [91]:
def hybrid_search(
    col,
    query_dense_embedding,
    query_sparse_embedding,
    sparse_weight=1.0,
    dense_weight=1.0,
    limit=5,
):
    dense_search_params = {"metric_type": "IP", "params": {}}
    dense_req = AnnSearchRequest(
        [query_dense_embedding], "dense_vector", dense_search_params, limit=limit
    )
    sparse_search_params = {"metric_type": "IP", "params": {}}
    sparse_req = AnnSearchRequest(
        [query_sparse_embedding], "sparse_vector", sparse_search_params, limit=limit
    )
    rerank = WeightedRanker(sparse_weight, dense_weight)
    res = col.hybrid_search(
        [sparse_req, dense_req], rerank=rerank, limit=limit, output_fields=["text"]
    )[0]
    return [hit.get("text") for hit in res]

In [92]:
dense_results = dense_search(col, query_embeddings["dense"][0])
sparse_results = sparse_search(col, query_embeddings["sparse"][[0]])
hybrid_results = hybrid_search(
    col,
    query_embeddings["dense"][0],
    query_embeddings["sparse"][[0]],
    sparse_weight=0.7,
    dense_weight=1.0,
)

/tmp/ipykernel_811/2991858840.py:16: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  res = col.search(
/tmp/ipykernel_811/3422049212.py:4: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  res = col.search(
/tmp/ipykernel_811/3693755873.py:18: PyMilvusDeprecationWarning: `Collection.hybrid_search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  res = col.hybrid_search(


In [93]:
print(dense_results)

['2\n', '3\nContents\nABOUT THIS REPORT ................................................................................................................................................. 4\nLegal sources and use of terminology ............................................................................................................... 4\nARTIFICIAL INTELLIGENCE AND DATA PROTECTION .....................................................................................................5\nHOW DOES ARTIFICIAL INTELLIGENCE WORK? ............................................................................................................ 7\nMachine learning ................................................................................................................................................. 7\nResults of learning ............................................................................................................................................. 10\nThe more training data the b

In [94]:
print(sparse_results)

['5\nArtificial intelligence and data\nprotection\nArtificial intelligence (AI) is the concept used to processing power and access to cheaper and greater\ndescribe computer systems that are able to learn from storage capacity. Big Data often refers to vast volumes\ntheir own experiences and solve complex problems in of data, extracted from multiple sources, often in real\ndifferent situations – abilities we previously thought time.4 These enormous data streams can be utilised\nwere unique to mankind. And it is data, in many cases for the benefit of society by means of analysis and\npersonal data, that fuels these systems, enabling them finding patterns and connections.\nto learn and become intelligent.\nThis is where AI can make a difference. While\nThe development of AI has made some major traditional analytical methods need to be\nadvances in recent years and its potential appears to programmed to find connections and links, AI learns\nbe promising: a better and more efficient public

In [95]:
print(hybrid_results)

['2\n', '3\nContents\nABOUT THIS REPORT ................................................................................................................................................. 4\nLegal sources and use of terminology ............................................................................................................... 4\nARTIFICIAL INTELLIGENCE AND DATA PROTECTION .....................................................................................................5\nHOW DOES ARTIFICIAL INTELLIGENCE WORK? ............................................................................................................ 7\nMachine learning ................................................................................................................................................. 7\nResults of learning ............................................................................................................................................. 10\nThe more training data the b

In [96]:
def doc_text_formatting(ef, query, docs):
    tokenizer = ef.model.tokenizer

    #tokenizer.encode(query, ...): Codifica la consulta en IDs de tokens.
    query_tokens_ids = tokenizer.encode(query, return_offsets_mapping=True)
    #query_tokens = ...: Convierte los IDs de tokens de la consulta de vuelta a
    #sus tokens legibles (palabras o subpalabras), que se usarán para la comparación.
    query_tokens = tokenizer.convert_ids_to_tokens(query_tokens_ids)
    formatted_texts = []

    for doc in docs:
        ldx = 0
        landmarks = []
        #Codifica el documento, obteniendo los IDs y,
        #las compensaciones (offsets), que indican la posición de inicio y fin
        #de cada token en la cadena de texto original.
        encoding = tokenizer.encode_plus(doc, return_offsets_mapping=True)
        tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])[1:-1]
        offsets = encoding["offset_mapping"][1:-1]
        for token, (start, end) in zip(tokens, offsets):
            #Verifica si el token actual del documento está presente en la lista de tokens de la consulta
            if token in query_tokens:
                # Si son iguales, significa que dos tokens de consulta coincidentes
                # son adyacentes en el texto del documento.
                if len(landmarks) != 0 and start == landmarks[-1]:
                    landmarks[-1] = end
                # Si no son adyacentes, registra un nuevo par de start y end en la lista landmarks.
                else:
                    landmarks.append(start)
                    landmarks.append(end)
        close = False
        formatted_text = ""
        for i, c in enumerate(doc):
            if ldx == len(landmarks):
                pass
            elif i == landmarks[ldx]:
                if close:
                    formatted_text += "</span>"
                else:
                    formatted_text += "<span style='color:red'>"
                close = not close
                ldx = ldx + 1
            formatted_text += c
        if close is True:
            formatted_text += "</span>"
        formatted_texts.append(formatted_text)
    return formatted_texts

In [97]:
from IPython.display import Markdown, display

# Redefinimos localmente una versión robusta y compatible de la función para corregir el error del tokenizador
def doc_text_formatting(ef, query, docs):
    tokenizer = ef.model.tokenizer
    query_tokens_ids = tokenizer.encode(query, add_special_tokens=False)
    query_tokens = tokenizer.convert_ids_to_tokens(query_tokens_ids)

    formatted_texts = []
    for doc in docs:
        # Usamos el llamador directo del tokenizador que genera el offset_mapping de forma nativa
        encoding = tokenizer(doc, return_offsets_mapping=True, add_special_tokens=False)
        tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
        offsets = encoding["offset_mapping"]

        landmarks = []
        for token, (start, end) in zip(tokens, offsets):
            if token in query_tokens:
                if len(landmarks) != 0 and start == landmarks[-1]:
                    landmarks[-1] = end
                else:
                    landmarks.append(start)
                    landmarks.append(end)

        ldx = 0
        formatted_text = ""
        close = False
        for i, c in enumerate(doc):
            if ldx == len(landmarks):
                pass
            elif i == landmarks[ldx]:
                if close:
                    formatted_text += "</span>"
                else:
                    formatted_text += "<span style='color:red'>"
                close = not close
                ldx += 1
            formatted_text += c
        if close:
            formatted_text += "</span>"
        formatted_texts.append(formatted_text)
    return formatted_texts

# Dense search results
display(Markdown("**Dense Search Results:**"))
formatted_results = doc_text_formatting(ef, query, dense_results)
for result in formatted_results:
    display(Markdown(result))

**Dense Search Results:**

2


3
Contents
ABOUT THIS REPORT ................................................................................................................................................. 4
Legal sources and use of terminology ............................................................................................................... 4
ARTIFICIAL INTELLIGENCE AND DATA PROTECTION .....................................................................................................5
HOW DOES ARTIFICIAL INTELLIGENCE WORK? ............................................................................................................ 7
Machine learning ................................................................................................................................................. 7
Results of learning ............................................................................................................................................. 10
The more training data the better?................................................................................................................... 11
The Black Box .................................................................................................................................................... 12
ARTIFICIAL INTELLIGENCE MEETS THE GDPR .......................................................................................................... 15
Fundamental principles of data protection ..................................................................................................... 15
Algorithmic bias meets the fairness principle ................................................................................................. 16
Artificial intelligence meets the principle of purpose limitation .................................................................... 16
Artificial intelligence meets data minimisation ............................................................................................... 18
The black box meets the principle of transparent processing ......................................................................... 19
CONTROLLING THE ALGORITHMS ............................................................................................................................ 23
The DPA’s supervisory competence ................................................................................................................ 23
Investigating the use of AI ............................................................................................................................... 23
How deep can an investigation go? ................................................................................................................. 23
How to inspect a “black box”? ......................................................................................................................... 24
SOLUTIONS AND RECOMMENDATIONS ...................................................................................................................... 25
Assess the data protection impact – and build privacy into your system! .................................................... 25
Tools and methods for good data protection in AI ......................................................................................... 26
Recommendations for privacy friendly development and use of AI ............................................................. 28


8
There are several forms of learning that can be used, other cats than those originally shown. In similar
depending on whether the information has been labelled fashion, a machine learning model will develop the same
or not. Labelled data is tagged data: if the data consists ability to recognise objects based on labelled images.
of images, the labels or tags may for example be gender,
ethnicity, dog or cat. If one is working with a dataset and wishes to separate
men and women, one can use different features that are
Below we have listed the main forms of learning, and we of relevance. The features used will depend on the basic
describe how the data is used in these. data available. For example, women live longer than
men on average, so life duration is of relevance when
Supervised learning differentiating between genders. This feature will,
however, prove to be somewhat narrow in most cases,
Supervised learning involves the use of labelled data, by and is mentioned here only as an example. If one’s data
means of which the supervision is performed. The basis consists of images, then hair length, or the use of
dataset is split into two, usually an 80/20 split, with 80 make-up or jewellery, may be relevant features. The
per cent of the data used to train the model. The example below illustrates how two different features are
remaining 20 per cent is used to verify how precisely the used in learning.
model processes unknown data. It is no good if the
model performs accurately using the training data and Learning takes place as follows (Illustrated by figure 2,
inaccurately using new and unknown data. If the model from left to right):
is too well adjusted to the training data, which we call
overfitting, it will not produce satisfactory results using 1. A set of labelled data is used.
new data. Therefore, the model requires a certain degree 2. Depending on data type, and what is considered
of generalisation. relevant, the features (circles and triangles) to
be used for learning are selected. The data is
Training data may for example consist of images labelled labelled to denote the right answer.
with information about the contents of each image. 3. A model is built that, based on the same
Supervised learning may be compared to teaching a features, will produce a label.
child. For example, we point to a number of objects to
the child and give them names. If we show a number of
cats to a child, the child will gradually learn to recognise


29
Recommendations for end users compelling and justifiable grounds for processing
the data, and that these grounds weigh more
These recommendations are aimed at end users. An end heavily than your interests, rights and freedoms.
user is the data subject using a service or whose personal  Right to demand limited processing. If you
details are processed by using AI. are of the opinion that some details are incorrect,
or are being processed unlawfully, or you have
 Right to information. You are entitled to exercised your right to protest against processing,
comprehensible and readily available information the organisation may be compelled to stop the
about the processing of your personal data. This data being used, but continue to store them until
right applies both when organisations retrieve the disagreement has been settled.
information directly from you, and when it is  Data portability. If, whether contractually or
retrieved from other sources. You shall know having given your consent, you have had personal
what the information is being used for (the data about yourself processed, you can ask for
purpose) and the legal basis on which that the these details to be delivered to you by the
organisation is processing the information. organisation in a structured, generally applicable,
 Consent. In many situations the controller must and machine-readable format.
obtain your consent before processing can begin.
The data controller is responsible for Recommendations for authorities
documenting that proper consent has been given,
which means that you have given a voluntary, These recommendations are for legislators and political
specific, informed and unambiguous declaration decision makers as they set the terms and conditions for
that you approve of your personal data being the development and use of AI.
processed. You also have the right to withdraw
any consent you have given previously.  Ensure that the public sector sets a good example
 Right of access to information. You have the in using AI. This requires acute awareness of the
right to contact organisations and ask whether ethical and privacy consequences of the systems
they are processing details about you, and, if so, they use, as well as expertise as buyers to make
what has been registered. As a rule, you are sure that the systems purchased have privacy by
entitled to a copy of the details registered. There design and that they meet the legislative
are, however, some exceptions to the right of requirements.
access to information, for example within the  Allocate funds to research that ensure the
judicial sector. technology processes the personal data in
 Right to rectify and delete information. compliance with the regulations. Protecting
You are entitled to ask for incorrect or personal data is not just a legal requirement, but
unnecessary details about you to be rectified or can also be a competitive advantage for
deleted. Norwegian industry.
 Right to object to the processing of details  Ensure that the enforcement authorities possess
about you. You may have the right to protest the relevant expertise, and arrange for experience
against the processing of details concerning and knowledge sharing across sectoral
yourself. If you protest against direct marketing, boundaries.
it must be stopped without your needing to  Ensure that the law keeps apace with
provide further grounds. In other situations, you technological developments. This applies to all
may have to justify your right to object by legislation that has relevance for the use of
explaining the circumstances affecting your personal data.
situation. The organisation must then cease
processing, unless they can prove they have


27
Microsoft for example has published a white paper on a were, in order to quality assure and improve their
system that uses homomorphic encryption in connection products.
with image recognition.20 Active efforts are also
underway to standardise homomorphic encryption Explainable AI (XAI)24
solutions.21 XAI is the idea that all the automated decisions made
should be explicable. With people involved in a process,
Transfer learning22 it is often desirable that an explanation is given for the
It is not the case that it is always necessary to develop outcome. There are some interesting possibilities in two
models from scratch. Another possibility is to utilise areas. There will also be a need to be able to control
existing models that solve similar tasks. By basing systems that do not have this embedded. It will probably
processing on these existing models, it will often be also be attractive for developers employing transfer
possible to achieve the same result with less data and in learning.
a shorter time. There are libraries containing pre-
trained models that can be used. There is also a project underway in this field, being run
by the Defense Advanced Research Projects Agency
RAIRD (DARPA), where the objective is to gain more knowledge
Statistics Norway (SSB) and the Norwegian Centre for about providing understandable explanations for
Research Data (NSD) have developed a system called automated decisions. They have sponsored Oregon State
RAIRD23 that permits research to be carried out on data University, awarding an amount of USD 6.5 million over
without having direct access to the complete dataset. four years for research into this topic. The goal is to
create AI that can explain its decisions in such a way
In short, this system works by means of an interface that that is understandable and promotes confidence in using
allows the researchers to access only the metadata of the the system. There are in any case good grounds for
underlying dataset. The dataset may, for example, be a believing that this research will drive the field forward.
cancer diagnosis register containing fields for age,
gender, date of and place of birth. The researcher can LIME25
then submit queries based on the metadata and obtain a LIME is an approach to XAI. It is a model-agnostic
report containing aggregated data only. solution that produces explanations ordinary people can
understand. In the case of image recognition, for
This solution has been designed to prevent the retrieval example, it will be able to show which parts of the
of data relating to very small groups and individual picture are relevant for what it thinks the image is. This
persons. This type of system can therefore be used when makes it easy for anyone to comprehend the basis for a
data for machine learning is needed. Instead of receiving decision.
a report as an end result, one could obtain a model from
the system.
3. Methods for avoidiing the black box issue
One of the issues mentioned is the lack of transparency
in connection with machine learning and automated
decision-making. This represents a challenge both for
those using such a system and for the people whose data
is processed by it. System developers who base their
work on machine learning would derive great benefit
from knowing what takes place under the bonnet, as it
20 https://www.microsoft.com/en-us/research/publication/cryptonets- 23 http://raird.no/
applying-neural-networks-to-encrypted-data-with-high-throughput-and-
accuracy/ 24 https://www.darpa.mil/program/explainable-artificial-intelligence
21 http://homomorphicencryption.org/ 25 https://www.oreilly.com/learning/introduction-to-local-interpretable-
model-agnostic-explanations-lime
22 http://www.cs.utexas.edu/~ml/publications/area/125/transfer_learning


In [98]:
# Sparse search results
display(Markdown("\n**Sparse Search Results:**"))
formatted_results = doc_text_formatting(ef, query, sparse_results)
for result in formatted_results:
    display(Markdown(result))


**Sparse Search Results:**

<span style='color:red'>5</span>
Artificial intelligence and data
protection
Artificial intelligence (AI) is the concept used to processing power and access to cheaper and greater
describe computer systems that are able to learn from storage capacity. Big Data often refers to vast volumes
their own experiences and solve complex problems in of data, extracted from multiple sources, often in real
different situations – abilities we previously thought time.4 These enormous data streams can be utilised
were unique to mankind. And it is data, in many cases for the benefit of society by means of analysis and
personal data, that fuels these systems, enabling them finding patterns and connections.
to learn and become intelligent.
This is where AI can make a difference. While
The development of AI has made some major traditional analytical methods need to be
advances in recent years and its potential appears to programmed to find connections and links, AI learns
be promising: a better and more efficient public from all the data it sees. Computer systems can
sector, new methods of climate and environmental therefore respond continuously to new data and
protection, a safer society, and perhaps even a cure adjust their analyses without human intervention.
for cancer. Thus, AI helps to remove the technical barriers that
traditional methods run into when analysing Big
We are in other words embarking on a venture that Data.
will without doubt have a considerable impact on
society. Accordingly, it is important for us to engage Greater demand for data, more stringent regulations
in discussion now. What sort of regulatory framework
do we need in order to grasp the opportunities offered The new data protection regulations that enters into
by AI in an assured and just manner? For we cannot force in May 2018 will strengthen our privacy rights,
escape the fact that the use of AI raises a number of while intensifying the requirements made of those
concerns with respect to ethics, security, legal processing such data. Organisations will bear more
responsibility, etc. This report is devoted to one such responsibility for processing personal data in
concern: the use of personal data in AI and the issue accordance with the regulation, and transparency
of privacy. requirements will be more stringent.
At the same time as the requirements are being
From winter to spring – why now?
intensified, demand for data is growing. AI-based
The concept of AI was known as far back as in the systems can become intelligent only if they have
1950s as a technology in which people had high hopes enough relevant data to learn from.
of success. The initial progress made was however
followed by many decades that are often called the AI An intelligent chatbot (a computer program that
winter because the early expectations were not met. people can interact with by means of ordinary speech,
In recent years, though, we have witnessed the or through written input) analyses all the information
coming of spring. it is fed – a combination of questions posed by
customers and responses communicated by customer
Today we see that AI is used to solve specific tasks service. From its analysis the chatbot can
such as, for example, image and speech recognition. “understand” what a customer is asking about and is
This is often called specialised AI. General AI refers therefore able to give a meaningful answer. The
to systems that are as versatile as humans when it greater the volume of information the chatbot can
comes to learning and problem solving. But it will base its analysis on, the better and more precise will
probably be several decades before this is achieved. be the reply it gives.
The AI spring has dawned thanks to the availability of
huge amounts of data, coupled with an increase in
4 https://ico.org.uk/for-organisations/guide-to-data-protection/big-data/


15
Artificial intelligence meets
the GDPR
The provisions of the GDPR govern the data controller’s
Personal data
duties and the rights of the data subject when personal
information is processed. The GDPR therefore applies
when artificial intelligence is under development with
Personal data means any information relating
the help of personal data, and also when it is used to
to an identified or identifiable natural person.
analyse or reach decisions about individuals.
(GDPR Article 4 (1))
In this chapter we will review the principles of data
The data may be directly linked to a person,
protection and the articles of the GDPR that are
such as a name, identification number or
especially relevant to the development and use of
location data.
artificial intelligence.
The data may also be indirectly linked to a
person. This means that the person can be
Fundamental principles of data
identified on the basis of a combination of one
protection or more elements that are specific to a person’s
physical, physiological, genetic, mental,
economic, cultural or social identity.
The rules governing the processing of personal data have
their basis in some fundamental principles. Article <span style='color:red'>5</span> of
the GDPR lists the principles that apply to all personal
data processing. The essence of these principles is that
personal information shall be utilised in a way that Processing
protects the privacy of the data subject in the best
possible way, and that each individual has the right to
decide how his or her personal data is used. The use of Processing means any operation or set of
personal data in the development of artificial operations which is performed on personal
intelligence challenges several of these principles. data, such as collection, recording,
organisation, structuring, storage, adaptation
In summary, these principles require that personal data or alteration, retrieval, consultation, use,
is: disclosure by transmission, dissemination or
otherwise making available, alignment or
 processed in a lawful, fair and transparent combination, restriction, erasure or
manner (principle of legality, fairness and destruction.
transparency)
 collected for specific, expressly stated and (GDPR Article 4 (2))
justified purposes and not treated in a new way
that is incompatible with these purposes
(principle of purpose limitation)
 adequate, relevant and limited to what is
Data controller
necessary for fulfilling the purposes for which it
is being processed (principle of data
minimisation)
Data controller means the natural or legal
 correct and, if necessary, updated (accuracy
person, public authority, agency or other body
principle)
which, alone or jointly with others, determines
 not stored in identifiable form for longer
the purposes and means of the processing of
periods than is necessary for the purposes
personal data.
(principle relating to data retention periods)
 processed in a way that ensures adequate
(GDPR Article 4 (7))
personal data protection (principle of integrity
and confidentiality)


In [99]:
# Hybrid search results
display(Markdown("\n**Hybrid Search Results:**"))
formatted_results = doc_text_formatting(ef, query, hybrid_results)
for result in formatted_results:
    display(Markdown(result))


**Hybrid Search Results:**

2


3
Contents
ABOUT THIS REPORT ................................................................................................................................................. 4
Legal sources and use of terminology ............................................................................................................... 4
ARTIFICIAL INTELLIGENCE AND DATA PROTECTION .....................................................................................................5
HOW DOES ARTIFICIAL INTELLIGENCE WORK? ............................................................................................................ 7
Machine learning ................................................................................................................................................. 7
Results of learning ............................................................................................................................................. 10
The more training data the better?................................................................................................................... 11
The Black Box .................................................................................................................................................... 12
ARTIFICIAL INTELLIGENCE MEETS THE GDPR .......................................................................................................... 15
Fundamental principles of data protection ..................................................................................................... 15
Algorithmic bias meets the fairness principle ................................................................................................. 16
Artificial intelligence meets the principle of purpose limitation .................................................................... 16
Artificial intelligence meets data minimisation ............................................................................................... 18
The black box meets the principle of transparent processing ......................................................................... 19
CONTROLLING THE ALGORITHMS ............................................................................................................................ 23
The DPA’s supervisory competence ................................................................................................................ 23
Investigating the use of AI ............................................................................................................................... 23
How deep can an investigation go? ................................................................................................................. 23
How to inspect a “black box”? ......................................................................................................................... 24
SOLUTIONS AND RECOMMENDATIONS ...................................................................................................................... 25
Assess the data protection impact – and build privacy into your system! .................................................... 25
Tools and methods for good data protection in AI ......................................................................................... 26
Recommendations for privacy friendly development and use of AI ............................................................. 28


8
There are several forms of learning that can be used, other cats than those originally shown. In similar
depending on whether the information has been labelled fashion, a machine learning model will develop the same
or not. Labelled data is tagged data: if the data consists ability to recognise objects based on labelled images.
of images, the labels or tags may for example be gender,
ethnicity, dog or cat. If one is working with a dataset and wishes to separate
men and women, one can use different features that are
Below we have listed the main forms of learning, and we of relevance. The features used will depend on the basic
describe how the data is used in these. data available. For example, women live longer than
men on average, so life duration is of relevance when
Supervised learning differentiating between genders. This feature will,
however, prove to be somewhat narrow in most cases,
Supervised learning involves the use of labelled data, by and is mentioned here only as an example. If one’s data
means of which the supervision is performed. The basis consists of images, then hair length, or the use of
dataset is split into two, usually an 80/20 split, with 80 make-up or jewellery, may be relevant features. The
per cent of the data used to train the model. The example below illustrates how two different features are
remaining 20 per cent is used to verify how precisely the used in learning.
model processes unknown data. It is no good if the
model performs accurately using the training data and Learning takes place as follows (Illustrated by figure 2,
inaccurately using new and unknown data. If the model from left to right):
is too well adjusted to the training data, which we call
overfitting, it will not produce satisfactory results using 1. A set of labelled data is used.
new data. Therefore, the model requires a certain degree 2. Depending on data type, and what is considered
of generalisation. relevant, the features (circles and triangles) to
be used for learning are selected. The data is
Training data may for example consist of images labelled labelled to denote the right answer.
with information about the contents of each image. 3. A model is built that, based on the same
Supervised learning may be compared to teaching a features, will produce a label.
child. For example, we point to a number of objects to
the child and give them names. If we show a number of
cats to a child, the child will gradually learn to recognise


29
Recommendations for end users compelling and justifiable grounds for processing
the data, and that these grounds weigh more
These recommendations are aimed at end users. An end heavily than your interests, rights and freedoms.
user is the data subject using a service or whose personal  Right to demand limited processing. If you
details are processed by using AI. are of the opinion that some details are incorrect,
or are being processed unlawfully, or you have
 Right to information. You are entitled to exercised your right to protest against processing,
comprehensible and readily available information the organisation may be compelled to stop the
about the processing of your personal data. This data being used, but continue to store them until
right applies both when organisations retrieve the disagreement has been settled.
information directly from you, and when it is  Data portability. If, whether contractually or
retrieved from other sources. You shall know having given your consent, you have had personal
what the information is being used for (the data about yourself processed, you can ask for
purpose) and the legal basis on which that the these details to be delivered to you by the
organisation is processing the information. organisation in a structured, generally applicable,
 Consent. In many situations the controller must and machine-readable format.
obtain your consent before processing can begin.
The data controller is responsible for Recommendations for authorities
documenting that proper consent has been given,
which means that you have given a voluntary, These recommendations are for legislators and political
specific, informed and unambiguous declaration decision makers as they set the terms and conditions for
that you approve of your personal data being the development and use of AI.
processed. You also have the right to withdraw
any consent you have given previously.  Ensure that the public sector sets a good example
 Right of access to information. You have the in using AI. This requires acute awareness of the
right to contact organisations and ask whether ethical and privacy consequences of the systems
they are processing details about you, and, if so, they use, as well as expertise as buyers to make
what has been registered. As a rule, you are sure that the systems purchased have privacy by
entitled to a copy of the details registered. There design and that they meet the legislative
are, however, some exceptions to the right of requirements.
access to information, for example within the  Allocate funds to research that ensure the
judicial sector. technology processes the personal data in
 Right to rectify and delete information. compliance with the regulations. Protecting
You are entitled to ask for incorrect or personal data is not just a legal requirement, but
unnecessary details about you to be rectified or can also be a competitive advantage for
deleted. Norwegian industry.
 Right to object to the processing of details  Ensure that the enforcement authorities possess
about you. You may have the right to protest the relevant expertise, and arrange for experience
against the processing of details concerning and knowledge sharing across sectoral
yourself. If you protest against direct marketing, boundaries.
it must be stopped without your needing to  Ensure that the law keeps apace with
provide further grounds. In other situations, you technological developments. This applies to all
may have to justify your right to object by legislation that has relevance for the use of
explaining the circumstances affecting your personal data.
situation. The organisation must then cease
processing, unless they can prove they have


27
Microsoft for example has published a white paper on a were, in order to quality assure and improve their
system that uses homomorphic encryption in connection products.
with image recognition.20 Active efforts are also
underway to standardise homomorphic encryption Explainable AI (XAI)24
solutions.21 XAI is the idea that all the automated decisions made
should be explicable. With people involved in a process,
Transfer learning22 it is often desirable that an explanation is given for the
It is not the case that it is always necessary to develop outcome. There are some interesting possibilities in two
models from scratch. Another possibility is to utilise areas. There will also be a need to be able to control
existing models that solve similar tasks. By basing systems that do not have this embedded. It will probably
processing on these existing models, it will often be also be attractive for developers employing transfer
possible to achieve the same result with less data and in learning.
a shorter time. There are libraries containing pre-
trained models that can be used. There is also a project underway in this field, being run
by the Defense Advanced Research Projects Agency
RAIRD (DARPA), where the objective is to gain more knowledge
Statistics Norway (SSB) and the Norwegian Centre for about providing understandable explanations for
Research Data (NSD) have developed a system called automated decisions. They have sponsored Oregon State
RAIRD23 that permits research to be carried out on data University, awarding an amount of USD 6.5 million over
without having direct access to the complete dataset. four years for research into this topic. The goal is to
create AI that can explain its decisions in such a way
In short, this system works by means of an interface that that is understandable and promotes confidence in using
allows the researchers to access only the metadata of the the system. There are in any case good grounds for
underlying dataset. The dataset may, for example, be a believing that this research will drive the field forward.
cancer diagnosis register containing fields for age,
gender, date of and place of birth. The researcher can LIME25
then submit queries based on the metadata and obtain a LIME is an approach to XAI. It is a model-agnostic
report containing aggregated data only. solution that produces explanations ordinary people can
understand. In the case of image recognition, for
This solution has been designed to prevent the retrieval example, it will be able to show which parts of the
of data relating to very small groups and individual picture are relevant for what it thinks the image is. This
persons. This type of system can therefore be used when makes it easy for anyone to comprehend the basis for a
data for machine learning is needed. Instead of receiving decision.
a report as an end result, one could obtain a model from
the system.
3. Methods for avoidiing the black box issue
One of the issues mentioned is the lack of transparency
in connection with machine learning and automated
decision-making. This represents a challenge both for
those using such a system and for the people whose data
is processed by it. System developers who base their
work on machine learning would derive great benefit
from knowing what takes place under the bonnet, as it
20 https://www.microsoft.com/en-us/research/publication/cryptonets- 23 http://raird.no/
applying-neural-networks-to-encrypted-data-with-high-throughput-and-
accuracy/ 24 https://www.darpa.mil/program/explainable-artificial-intelligence
21 http://homomorphicencryption.org/ 25 https://www.oreilly.com/learning/introduction-to-local-interpretable-
model-agnostic-explanations-lime
22 http://www.cs.utexas.edu/~ml/publications/area/125/transfer_learning


In [100]:
final_template = f"""
You are an expert assistant providing detailed and accurate information. The user has asked the following question:
**Question:** {query}

Below is relevant information retrieved from various sources:
**Retrieved Information:**
{hybrid_results}

**Instruction:** Use only the query-relevant content from the retrieved information to answer the question. Focus on providing a comprehensive, informative response based solely on the given data. If any conflicting details are present, prioritize the most reliable and consistent information.
"""
final_template

"\nYou are an expert assistant providing detailed and accurate information. The user has asked the following question:\n**Question:** 5\n\nBelow is relevant information retrieved from various sources:\n**Retrieved Information:**\n['2\\n', '3\\nContents\\nABOUT THIS REPORT ................................................................................................................................................. 4\\nLegal sources and use of terminology ............................................................................................................... 4\\nARTIFICIAL INTELLIGENCE AND DATA PROTECTION .....................................................................................................5\\nHOW DOES ARTIFICIAL INTELLIGENCE WORK? ............................................................................................................ 7\\nMachine learning ..........................................................................................................

In [101]:
!pip install langchain_groq

In [104]:
from groq import Groq
from google.colab import userdata

client = Groq(
    api_key=userdata.get('GROQ_API_KEY'),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": final_template,
        }
    ],
    #Modelo-1:
    model="openai/gpt-oss-20b",

    #Modelo-2:
    #model="openai/gpt-oss-120b",
)

print(chat_completion.choices[0].message.content)

**5 – “Artificial Intelligence and Data Protection”**  
*(This is the fifth main heading of the report, appearing on page 5 of the contents list.)*

---

### 1.  Scope of the section  
- **Legal foundation** – The report begins by laying out the *legal sources* that govern the use of AI, with a focus on the General Data Protection Regulation (GDPR) and related European legislation.  
- **Terminology** – It clarifies key terms such as *algorithm*, *black box*, *explainable AI (XAI)*, *transfer learning*, and *homomorphic encryption*, so readers know exactly what is meant when those concepts appear later in the analysis.  

### 2.  How AI Works  
- **Machine learning basics** – AI systems learn by analysing labelled data.  
  - *Supervised learning* uses a training set split (typically 80 / 20) to build a model that predicts labels (e.g., gender, ethnicity, object identity).  
  - *Over‑fitting* occurs if a model performs well on training data but poorly on new data, highlighting the nee